# Introduction to MLOps: From Notebook to ML Application

<a href="https://colab.research.google.com/github/rubenfonnegra/machine_learning/blob/master/Sem_12/MLOps_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> 
<a href="https://github.com/rubenfonnegra/machine_learning/blob/master/Sem_12/MLOps_Colab.ipynb" target="_parent"><img src="https://img.shields.io/badge/%E2%80%8B-Open%20in%20Github-blue?logo=github" alt="Open In Github"/></a> 


### Learning Objectives

- Explain the main stages of the Machine Learning lifecycle, from training to monitoring (MLOps).
- Use MLflow to record, compare, and manage Machine Learning experiments.
- Save and reload trained Machine Learning pipelines using Joblib.
- Create user-friendly Machine Learning interfaces with Gradio.
- Develop and test REST prediction endpoints using FastAPI, Uvicorn, and HTTPX.
- Detect potential data drift by comparing training and incoming data distributions.
- Recognize the importance of reproducibility, versioning, validation, security, and monitoring.


### Documentation

- [```MLflow```](https://mlflow.org/docs/latest/ml/tracking/)
- [```Gradio```](https://www.gradio.app/guides/quickstart)
- [```FastAPI```](https://fastapi.tiangolo.com/tutorial/)
- [```Uvicorn```](https://www.uvicorn.org/)
- [```Joblib```](https://joblib.readthedocs.io/en/latest/)
- [```Shutil```](https://docs.python.org/3/library/shutil.html)
- [```HTTPX```](https://www.python-httpx.org/quickstart/)
---

> **📘 Machine Learning**  
> **Author:** Rubén D. Fonnegra, Ph.D. \
> **Institution:** Institución Universitaria Pascual Bravo  
> © 2026 · Educational use with attribution


## Notebook roadmap

**MLOps** applies engineering practices to the ML lifecycle. 
- **Training** fits parameters using labeled examples
- **inference** uses the fitted model to predict on new inputs. 
- **Deployment** makes a model available in an environment
- **serving** exposes predictions through an interface or API. 

Reproducibility, model versioning, and monitoring help us understand what was trained, which version is running, and whether its inputs or quality have changed.

## The MLOps workflow

```text
Data → Training →   Evaluation.  → Model Artifact → Serving → Monitoring
                ↘ Experiment track ↗              ↘ Feedback / retraining
```

An **artifact** is a saved file such as a fitted pipeline or its metadata. This lesson uses local files and a local HTTP service, not permanent production deployment.

## Environment and dependencies

Install the packages not guaranteed in Colab. 
- MLflow records experiments locally
- Gradio builds a browser UI
- FastAPI and Uvicorn provide an HTTP service
- HTTPX sends test requests. 

The notebook uses current, maintained public APIs and avoids MLflow's optional dashboard.

In [ ]:
%pip -q install "mlflow>=3,<4" "gradio>=5,<7" "fastapi>=0.115,<1" "uvicorn>=0.30,<1" "httpx>=0.27,<1"

In [ ]:
import os, sys, json, time, random, socket, subprocess, importlib.metadata as metadata

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from sklearn.pipeline import Pipeline

import joblib
import mlflow
import httpx

SEED = 42
random.seed(SEED); np.random.seed(SEED)

ROOT = Path('mlops_wine')
ROOT.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ROOT / 'wine_pipeline.joblib'
META_PATH  = ROOT / 'model_metadata.json'

print('Workspace:', ROOT)

## Load and explore the Wine dataset

The dataset has **178 observations**, **13 numeric chemical measurements**, and **3 wine classes**. We will use the exact feature order provided by scikit-learn for training, the Gradio form, and the API. The labels are dataset categories, not subjective wine quality ratings.

In [ ]:
wine = load_wine(as_frame=True)

X, y = wine.data.copy(), wine.target.copy()
feature_names, class_names = list(wine.feature_names), list(wine.target_names)

display(X.head(3))
print('Shape:', X.shape, '| Classes:', dict(zip(range(len(class_names)), class_names)))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
y.value_counts().sort_index().plot.bar(ax=axes[0], title='Class balance')
axes[1].scatter(X['alcohol'], X['color_intensity'], c=y, cmap='viridis', alpha=0.8)
axes[0].set(xlabel='Class index', ylabel='Samples'); axes[1].set(xlabel='Alcohol', ylabel='Color intensity', title='Two input features')

plt.tight_layout()
plt.show()

## Train and evaluate candidate pipelines

A scikit-learn `Pipeline` keeps preprocessing and classification together. The scaler learns statistics **only from training data**, preventing test-set leakage. Logistic Regression and Random Forest are deliberately simple baselines. We use accuracy and macro F1 because macro F1 weights all classes equally.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)

candidates = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=SEED), {'max_iter': 1000}),
    'Random Forest': (RandomForestClassifier(n_estimators=100, random_state=SEED), {'n_estimators': 100})
}

trained, records = {}, []

for name, (classifier, params) in candidates.items():
    pipeline = Pipeline([('scaler', StandardScaler()), ('classifier', classifier)])
    pipeline.fit(X_train, y_train)

    predicted = pipeline.predict(X_test)

    trained[name] = pipeline
    records.append({'model': name, 
                    'accuracy': accuracy_score(y_test, predicted), 
                    'macro_f1': f1_score(y_test, predicted, average='macro'), **params})


In [ ]:
results = pd.DataFrame(records).sort_values(['macro_f1', 'accuracy'], ascending=False).reset_index(drop=True)
display(results.fillna('-'))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, name in zip(axes, candidates):
    cm = confusion_matrix(y_test, trained[name].predict(X_test))
    sns.heatmap(cm, annot=True, ax=ax, cmap='Blues')
    ax.set_title(name)

plt.tight_layout()
plt.show()

## Track experiments with MLflow

**Experiment tracking** records which configuration produced which metrics and artifacts. It is **not deployment**: logging a model does not make a prediction endpoint available. We use a local `file://` tracking store and MLflow's Python API so no server or account is necessary. Re-running the tracking cell creates new runs in the same experiment.

In [ ]:
TRACKING_DB = ROOT / "mlflow.db"

mlflow.set_tracking_uri(f"sqlite:///{TRACKING_DB.resolve()}")
mlflow.set_experiment("Wine Classification - Intro to MLOps")

run_ids = {}
for name, (classifier, params) in candidates.items():
    row = next(row for row in records if row['model'] == name)
    with mlflow.start_run(run_name=name) as run:
        mlflow.log_param('model_name', name)
        mlflow.log_param('scaler', 'StandardScaler')
        mlflow.log_param('random_seed', SEED)
        mlflow.log_params(params)
        mlflow.log_metrics({'accuracy': float(row['accuracy']), 'macro_f1': float(row['macro_f1'])})
        artifact = ROOT / f"{name.lower().replace(' ', '_')}.joblib"
        joblib.dump(trained[name], artifact)
        mlflow.log_artifact(str(artifact), artifact_path='pipeline')
        mlflow.set_tag('purpose', 'classroom demonstration')
        run_ids[name] = run.info.run_id

print('Logged run IDs:', run_ids)

### Compare tracked runs and retrieve the selected artifact

We choose the candidate with the highest **macro F1** (accuracy breaks ties). The run ID uniquely identifies the logged experiment run. The selected model is downloaded from **that run's artifact store**, then copied to the canonical model path used by both serving demonstrations.

In [ ]:
experiment = mlflow.get_experiment_by_name('Wine Classification - Intro to MLOps')
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

tracked = runs_df[['run_id', 'tags.mlflow.runName', 'metrics.accuracy', 'metrics.macro_f1']].rename(
    columns={'tags.mlflow.runName': 'model', 'metrics.accuracy': 'accuracy', 'metrics.macro_f1': 'macro_f1'})

display(tracked.sort_values('macro_f1', ascending=False).head(10))
tracked.drop_duplicates('model').set_index('model')[['accuracy', 'macro_f1']].plot.bar(figsize=(8, 3.5), ylim=(0, 1.05), rot=0, title='Tracked test metrics')
plt.ylabel('Score')
plt.tight_layout()
plt.show()

selected_name = str(results.loc[0, 'model'])
selected_run_id = run_ids[selected_name]

artifact_filename = f"{selected_name.lower().replace(' ', '_')}.joblib"
downloaded = mlflow.artifacts.download_artifacts(run_id=selected_run_id, artifact_path=f'pipeline/{artifact_filename}', dst_path=str(ROOT / 'retrieved'))
print('Selected model:', selected_name, '| Run ID:', selected_run_id)
print('Retrieved artifact:', downloaded)

## Save and reload the complete inference pipeline

`joblib` stores the **fitted scaler and classifier together**, preventing accidental use of an unscaled input at inference time. Its underlying pickle-based format can execute code when loaded: **never load joblib/pickle artifacts from untrusted sources**. Here, every artifact was created by our own notebook.

In [ ]:
import shutil

shutil.copy2(downloaded, MODEL_PATH)
model = joblib.load(MODEL_PATH)
original_predictions = trained[selected_name].predict(X_test)
reloaded_predictions = model.predict(X_test)

assert np.array_equal(original_predictions, reloaded_predictions), 'Reloaded predictions differ.'

versions = {name: metadata.version(name) for name in ['scikit-learn', 'numpy', 'joblib', 'mlflow']}
metadata_info = {'model_name': selected_name, 'run_id': selected_run_id, 'feature_names': feature_names, 'class_names': class_names, 'class_ids': [int(v) for v in model.classes_], 'versions': versions, 'random_seed': SEED}

META_PATH.write_text(json.dumps(metadata_info, indent=2), encoding='utf-8')
print('Saved pipeline:', MODEL_PATH)
print('Predictions identical:', np.array_equal(original_predictions, reloaded_predictions))

display(pd.DataFrame([{'actual': class_names[int(a)], 'predicted': class_names[int(p)]} for a, p in zip(y_test.iloc[:5], reloaded_predictions[:5])]))
print('Metadata:', META_PATH)

## Serve predictions through a Gradio application

**Gradio** converts Python functions into an interactive web interface. This app loads the **saved** pipeline; it does not retrain. All 13 numeric inputs use median values from the training split as defaults. Class probabilities come from `predict_proba`, and invalid/non-finite inputs produce a readable error.

**Colab note:** `share=True` requests a temporary, publicly accessible Gradio link. Anyone with that link may access the demonstration, so do not use sensitive data. The link expires and is **not permanent production hosting**. To avoid creating a public link, set `share=False`; local rendering may not be accessible from every Colab environment.

In [ ]:
import gradio as gr

app_model = joblib.load(MODEL_PATH)
app_meta = json.loads(META_PATH.read_text(encoding='utf-8'))
feature_defaults = X_train[feature_names].median().to_numpy(dtype=float)

def predict_wine(*values):
    try:
        numbers = np.asarray(values, dtype=float)
        if numbers.shape != (len(feature_names),) or not np.isfinite(numbers).all():
            raise ValueError('All 13 features must be finite numeric values.')
        sample = pd.DataFrame([numbers], columns=feature_names)
        predicted_id = int(app_model.predict(sample)[0])
        probabilities = app_model.predict_proba(sample)[0]
        distribution = {app_meta['class_names'][int(class_id)]: float(prob) for class_id, prob in zip(app_model.classes_, probabilities)}
        return app_meta['class_names'][predicted_id], distribution
    except (ValueError, TypeError) as exc:
        raise gr.Error(f'Invalid input: {exc}')


inputs = [gr.Number(label=feature.replace('_', ' ').title(), value=float(default), precision=4) for feature, default in zip(feature_names, feature_defaults)]
with gr.Blocks(title='Wine Classification | MLOps Demo') as demo:
    gr.Markdown('# Wine Classification\nPredict a wine category using the saved ML pipeline. Adjust the chemical measurements and select **Predict**.')
    with gr.Row():
        with gr.Column():
            for control in inputs: control.render()
            submit = gr.Button('Predict', variant='primary')
        with gr.Column():
            prediction_label = gr.Textbox(label='Predicted class')
            probability_chart = gr.Label(label='Class probabilities', num_top_classes=3)
    submit.click(fn=predict_wine, inputs=inputs, outputs=[prediction_label, probability_chart])

print('Gradio interface created. Run the next cell to launch it.')

### Launch Gradio

This cell may create a temporary public URL. Run it only when ready to share the classroom demonstration. If re-running, close the prior instance first. If Colab reports a proxy or localhost issue, use `share=True`.

In [ ]:
demo.launch(share=True, prevent_thread_lock=True, show_error=True)

In [ ]:
demo.close()

## Serve the same model through FastAPI

An **API** accepts structured requests from software clients rather than displaying a form. Here `GET /health` returns service status; `POST /predict` receives 13 features and returns JSON. **HTTP 200** means success; **422** means the request failed schema validation; **500** indicates an unexpected server-side error. Pydantic checks the request schema and rejects extra fields and non-finite values.

We generate a standalone `app.py` that loads the **same** saved joblib pipeline once at application startup. The API uses the metadata feature order to construct its input DataFrame.

In [ ]:
APP_SOURCE = r''' \
from pathlib import Path
import json, math, os
import joblib, pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict, create_model, field_validator

ROOT = Path(os.environ.get('WINE_MLOPS_DIR', Path(__file__).resolve().parent))
meta = json.loads((ROOT / 'model_metadata.json').read_text(encoding='utf-8'))
model = joblib.load(ROOT / 'wine_pipeline.joblib')
feature_names = meta['feature_names']

class FeatureValue(BaseModel):
    model_config = ConfigDict(extra='forbid')
    @field_validator('*')
    @classmethod
    def check_finite(cls, value):
        if not math.isfinite(value): raise ValueError('Features must be finite numbers')
        return value

WineInput = create_model('WineInput', __base__=FeatureValue, **{name: (float, ...) for name in feature_names})
app = FastAPI(title='Wine Classification API', version='1.0.0')

@app.get('/health')
def health():
    return {'status': 'ok', 'model_name': meta['model_name'], 'run_id': meta['run_id']}

@app.post('/predict')
def predict(payload: WineInput):
    try:
        values = payload.model_dump()
        sample = pd.DataFrame([[values[name] for name in feature_names]], columns=feature_names)
        class_id = int(model.predict(sample)[0])
        probabilities = model.predict_proba(sample)[0]
        return {'predicted_class': meta['class_names'][class_id], 'class_id': class_id, 'probabilities': {meta['class_names'][int(i)]: float(p) for i, p in zip(model.classes_, probabilities)}}
    except Exception as exc:
        raise HTTPException(status_code=500, detail='Prediction failed') from exc
'''
APP_PATH = ROOT / 'app.py'
APP_PATH.write_text(APP_SOURCE, encoding='utf-8')
print('Standalone FastAPI app saved:', APP_PATH)
print('To deploy elsewhere: copy app.py, wine_pipeline.joblib, model_metadata.json and install compatible dependencies.')

### Start a local Uvicorn server

Colab runs a background process that listens only on `127.0.0.1`, so **no public tunnel or API key** is required. Requests in the following cells originate from the same Colab runtime. If you rerun this cell, it stops the prior server process created by this notebook. The process is also stopped in the cleanup cell.

In [ ]:
if 'server_process' in globals() and server_process.poll() is None:
    server_process.terminate()
    try:
      server_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
      server_process.kill(); server_process.wait()

with socket.socket() as sock:
    sock.bind(('127.0.0.1', 0))
    PORT = sock.getsockname()[1]

BASE_URL = f'http://127.0.0.1:{PORT}'
env = os.environ.copy(); env['WINE_MLOPS_DIR'] = '/content/' + str(ROOT)
server_log = open(ROOT / 'uvicorn.log', 'w', encoding='utf-8')
server_process = subprocess.Popen([sys.executable, '-m', 'uvicorn',
                                   'app:app', '--host', '127.0.0.1',
                                   '--port', str(PORT)], cwd=str(ROOT),
                                   env=env, stdout=server_log,
                                   stderr=subprocess.STDOUT)


for attempt in range(50):
    if server_process.poll() is not None:
      raise RuntimeError((ROOT / 'uvicorn.log').read_text()[-3000:])
    try:
        response = httpx.get(BASE_URL + '/health', timeout=1)
        if response.status_code == 200: break
    except httpx.RequestError: pass
    time.sleep(0.2)
else: raise RuntimeError('Server did not start. Check ' + str(ROOT / 'uvicorn.log'))
print('Local API:', BASE_URL)
print('Health response:', response.json())

### Send valid and invalid HTTP requests

`httpx` acts as a software client. The valid request uses a held-out test observation; the invalid request deliberately omits a required feature. The service should return **200** and **422**, respectively. A third request sends an invalid string to demonstrate type validation.

In [ ]:
payload = {name: float(X_test.iloc[0][name]) for name in feature_names}
incomplete_payload = {k: v for k, v in payload.items() if k != feature_names[0]}
incorrect_payload  = {**payload, feature_names[0]: 'NaN'}

valid_response = httpx.post(BASE_URL + '/predict', json=payload, timeout=10)
missing_response = httpx.post(BASE_URL + '/predict', json=incomplete_payload, timeout=10)
wrong_type_response = httpx.post(BASE_URL + '/predict', json=incorrect_payload, timeout=10)

print('Valid request:', valid_response.status_code); print(json.dumps(valid_response.json(), indent=2))
print('Missing field:', missing_response.status_code, '| Error:', missing_response.json()['detail'][0]['type'])
print('Wrong type:', wrong_type_response.status_code, '| Error:', wrong_type_response.json()['detail'][0]['type'])

### Stop the local API server

Run this after finishing API experiments. If you want to test the API again, rerun the server-start cell.

In [ ]:
if 'server_process' in globals() and server_process.poll() is None:
    server_process.terminate()
    try: server_process.wait(timeout=5)
    except subprocess.TimeoutExpired: server_process.kill(); server_process.wait()
if 'server_log' in globals() and not server_log.closed: server_log.close()
print('Local API server stopped.')

## Production considerations

| Practice | Why it matters beyond Colab |
|---|---|
| Docker and dependency management | Recreate the runtime consistently |
| Model versioning | Trace each prediction to a specific model and training run |
| Input validation | Reject missing, malformed, or implausible values |
| Logging and error handling | Diagnose failures without exposing sensitive internals |
| API security | Add authentication, authorization, TLS, and rate limits |
| Monitoring | Track latency, failures, input drift, and quality when labels arrive |
| Retraining | Evaluate updated models before promoting them |
| CI/CD | Test and release changes predictably |

## Challenge: From Notebook to ML Application

Reuse the Wine dataset, feature order, and saved-pipeline approach. Do not create a separate unrelated model.

- Train a third classifier (for example, K-Nearest Neighbors or Support Vector Machine) inside a preprocessing pipeline, and log its parameters and metrics as an additional MLflow run.
- Compare all three candidates using accuracy and macro F1. Explain your selection rule and identify the selected run ID.
- Save and reload the selected complete pipeline; verify identical predictions.
- Modify the Gradio application: add an informative description or a reset/example interaction, while retaining all 13 inputs and probabilities.
- Restart the FastAPI service if needed and send a new valid observation to `/predict`; record the JSON response. Also test one invalid request.
- Identify two production risks and propose one mitigation for each.

---

## 📄 Attribution

This notebook was developed by **Rubén D. Fonnegra** as educational material for Machine Learning courses at **Institución Universitaria Pascual Bravo**.
You may use, share, and adapt this material for educational purposes, provided that appropriate credit is given to the original author.

**Suggested citation:**
> Fonnegra Tarazona, R. D. (2026). *Introduction to MLOps - From Notebook to ML Application: Machine Learning Notebook*. Institución Universitaria Pascual Bravo.